# Predictive Maintenance Data Pipeline (Refactored)

This notebook demonstrates the usage of the refactored predictive maintenance package.
It follows the same workflow as the original `data_pipeline.ipynb` but uses the new modular package structure.


## Import Dependencies


In [ ]:
import pandas as pd
import numpy as np
import json
import sys
import os

# Add the project root to Python path
sys.path.append('..')

# Import from the refactored package
from inverter_predictive_maintenance.preprocess import (
    load_parquet_data, 
    load_failure_sessions,
    missing_value_imputation,
    downsample_inverter_raw,
    exclude_periods_from_data,
    prepare_dataset
)

from inverter_predictive_maintenance.visualize import (
    visualize_mean_values,
    visualize_failure_timeline
)

print("[SUCCESS] All imports successful!")


## Configuration Parameters


In [ ]:
# Visualization control
perform_visualization = True

# Pre-failure labeling parameters
pre_days = 5

# Feature columns for analysis
feature_cols = [
    "metric.STATUS_AC_MOD_ADMISSION_TEMP.MEASURED",  # ambient temperature
    "metric.STATUS_INTERNAL_TEMP.MEASURED",          # internal temperature
    "metric.AC_VOLTAGE_AB.MEASURED",                 # AC voltage
    "metric.AC_VOLTAGE_BC.MEASURED",                 # AC voltage
    "metric.AC_VOLTAGE_CA.MEASURED",                 # AC voltage
    "metric.DC_VOLTAGE.MEASURED",                    # DC voltage
    "metric.AC_POWER.MEASURED",                      # AC power
]

# Periods to exclude due to data quality issues
exclude_periods = [
    [pd.Timestamp('2021-01-01'), pd.Timestamp('2021-12-23')], # data collection issue
    [pd.Timestamp('2023-02-23'), pd.Timestamp('2023-08-26')], # anomalies in the data
]

print(f"Configuration loaded:")
print(f"   - Pre-failure days: {pre_days}")
print(f"   - Features: {len(feature_cols)} columns")
print(f"   - Exclude periods: {len(exclude_periods)} periods")
print(f"   - Visualization: {'Enabled' if perform_visualization else 'Disabled'}")


## Data Loading

Load inverter data and failure sessions using the refactored data loading functions.


In [ ]:
# Load data using refactored functions
print("Loading data...")

try:
    inverter_data = load_parquet_data('../dataset/inverter_data')
    print(f"Loaded inverter data: {inverter_data.shape}")
except Exception as e:
    print(f"❌ Error loading inverter data: {e}")
    # Fallback to CSV if parquet fails
    inverter_data = pd.read_csv('../dataset/train_data.csv')
    print(f"✅ Loaded inverter data from CSV: {inverter_data.shape}")

try:
    failure_sessions = load_failure_sessions(
        '../dataset/failure_sessions_w_maintenance.csv', 
        min_days=3
    )
    print(f"✅ Loaded failure sessions: {failure_sessions.shape}")
except Exception as e:
    print(f"❌ Error loading failure sessions: {e}")
    failure_sessions = pd.DataFrame()

print(f"\n📊 Data Summary:")
print(f"   - Inverter data: {inverter_data.shape[0]:,} rows, {inverter_data.shape[1]} columns")
print(f"   - Failure sessions: {failure_sessions.shape[0]:,} sessions")
print(f"   - Date range: {inverter_data['event_local_time'].min()} to {inverter_data['event_local_time'].max()}")


## Data Visualization

Visualize failure sessions timeline and raw data patterns.


In [ ]:
if perform_visualization and not failure_sessions.empty:
    print("📈 Creating failure timeline visualization...")
    visualize_failure_timeline(failure_sessions)
else:
    print("⏭️ Skipping failure timeline visualization")


In [ ]:
if perform_visualization:
    print("📈 Creating raw data visualization...")
    try:
        visualize_mean_values(
            inverter_data, 
            failure_sessions, 
            feature_cols, 
            '../plot', 
            'Raw Data Visualization'
        )
        print("✅ Raw data visualization completed")
    except Exception as e:
        print(f"❌ Error creating visualization: {e}")
else:
    print("⏭️ Skipping raw data visualization")


## Data Preprocessing

Apply comprehensive data preprocessing using the refactored functions.


In [ ]:
# Filter data to only include required columns
print("🔧 Filtering data columns...")
filtered_data = inverter_data[['event_local_time', 'device_name'] + feature_cols].copy()
print(f"✅ Filtered data: {filtered_data.shape}")


### Anomaly Detection and Removal


In [ ]:
# Remove temperature anomalies
print("🔍 Detecting anomalies...")
if "metric.STATUS_AC_MOD_ADMISSION_TEMP.MEASURED" in filtered_data.columns:
    anomaly_mask = filtered_data["metric.STATUS_AC_MOD_ADMISSION_TEMP.MEASURED"] >= 100
    filtered_data.loc[anomaly_mask, "metric.STATUS_AC_MOD_ADMISSION_TEMP.MEASURED"] = None
    print(f"✅ Removed {anomaly_mask.sum():,} temperature anomalies")
else:
    print("⚠️ Temperature column not found")


### Missing Value Imputation


In [ ]:
print("🔧 Performing missing value imputation...")
try:
    imputed_df = missing_value_imputation(
        filtered_data, 
        feature_cols, 
        time_col='event_local_time', 
        device_col='device_name', 
        short_gap_limit=0, 
        long_fill_value=0.0, 
        add_missing_mask=True
    )
    
    # Extend feature columns to include missing masks
    extended_feature_cols = feature_cols + [col + '_missing' for col in feature_cols]
    print(f"✅ Missing value imputation completed")
    print(f"   - Extended features: {len(extended_feature_cols)} columns")
    
except Exception as e:
    print(f"❌ Error in missing value imputation: {e}")
    imputed_df = filtered_data
    extended_feature_cols = feature_cols


### Data Downsampling


In [ ]:
print("📉 Performing data downsampling...")
try:
    downsampled_data = downsample_inverter_raw(
        imputed_df, 
        freq='30T',  # 30-minute intervals
        drop_empty_bins=False
    )
    
    # Remove NaN values generated by downsampling
    initial_rows = len(downsampled_data)
    downsampled_data.dropna(inplace=True)
    final_rows = len(downsampled_data)
    
    print(f"✅ Downsampling completed")
    print(f"   - Initial rows: {initial_rows:,}")
    print(f"   - Final rows: {final_rows:,}")
    print(f"   - Removed: {initial_rows - final_rows:,} rows")
    
except Exception as e:
    print(f"❌ Error in downsampling: {e}")
    downsampled_data = imputed_df


### Data Labeling


In [ ]:
print("🏷️ Applying failure labels...")
try:
    labeled_data = prepare_dataset(
        downsampled_data, 
        failure_sessions, 
        pre_days=pre_days
    )
    
    # Check label distribution
    label_counts = labeled_data['label'].value_counts()
    print(f"✅ Data labeling completed")
    print(f"   - Total samples: {len(labeled_data):,}")
    print(f"   - Normal samples (0): {label_counts.get(0, 0):,}")
    print(f"   - Pre-failure samples (1): {label_counts.get(1, 0):,}")
    print(f"   - Pre-failure rate: {label_counts.get(1, 0) / len(labeled_data):.4f}")
    
except Exception as e:
    print(f"❌ Error in data labeling: {e}")
    labeled_data = downsampled_data
    labeled_data['label'] = 0  # Default to normal


### Feature Engineering


In [ ]:
print("⚙️ Engineering features...")

# Cyclical encoding for time features
labeled_data['month_sin'] = np.sin(2 * np.pi * labeled_data['event_local_time'].dt.month / 12)
labeled_data['month_cos'] = np.cos(2 * np.pi * labeled_data['event_local_time'].dt.month / 12)
labeled_data['hour_sin'] = np.sin(2 * np.pi * labeled_data['event_local_time'].dt.hour / 24)
labeled_data['hour_cos'] = np.cos(2 * np.pi * labeled_data['event_local_time'].dt.hour / 24)

# Voltage features
voltage_cols = ['metric.AC_VOLTAGE_AB.MEASURED', 'metric.AC_VOLTAGE_BC.MEASURED', 'metric.AC_VOLTAGE_CA.MEASURED']
if all(col in labeled_data.columns for col in voltage_cols):
    v_data = labeled_data[voltage_cols]
    labeled_data['V_mean'] = v_data.mean(axis=1)
    labeled_data['V_unbalance'] = (v_data.max(axis=1) - v_data.min(axis=1)) / (v_data.mean(axis=1) + 1e-6)

# Temperature delta
temp_cols = ['metric.STATUS_INTERNAL_TEMP.MEASURED', 'metric.STATUS_AC_MOD_ADMISSION_TEMP.MEASURED']
if all(col in labeled_data.columns for col in temp_cols):
    labeled_data['T_delta'] = labeled_data['metric.STATUS_INTERNAL_TEMP.MEASURED'] - labeled_data['metric.STATUS_AC_MOD_ADMISSION_TEMP.MEASURED']

# Update feature list
new_features = ['hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'V_mean', 'V_unbalance', 'T_delta']
extended_feature_cols.extend([f for f in new_features if f in labeled_data.columns])

print(f"✅ Feature engineering completed")
print(f"   - Total features: {len(extended_feature_cols)}")
print(f"   - New features: {[f for f in new_features if f in labeled_data.columns]}")


## Data Cleaning - Exclude Problematic Periods


In [ ]:
print("🧹 Excluding problematic periods...")

# Clean failure sessions
if not failure_sessions.empty:
    print(f"   - Initial failure sessions: {failure_sessions.shape[0]:,}")
    
    # Apply exclusion to failure sessions
    failure_sessions['event_local_time'] = failure_sessions['start_time']
    filtered_sessions = exclude_periods_from_data(failure_sessions, exclude_periods)
    filtered_sessions['event_local_time'] = filtered_sessions['end_time']
    filtered_sessions = exclude_periods_from_data(filtered_sessions, exclude_periods)
    
    print(f"   - Filtered failure sessions: {filtered_sessions.shape[0]:,}")
else:
    filtered_sessions = failure_sessions

# Clean inverter data
print(f"   - Initial inverter data: {labeled_data.shape[0]:,} rows")
labeled_data = exclude_periods_from_data(labeled_data, exclude_periods)
print(f"   - Filtered inverter data: {labeled_data.shape[0]:,} rows")

print("✅ Data cleaning completed")


## Dataset Splitting

Split the processed data into train, validation, and test sets.


In [ ]:
print("✂️ Splitting dataset...")

# Define split points (same as original data_pipeline.ipynb)
split_time = [pd.Timestamp('2024-06-30'), pd.Timestamp('2025-01-01')]

# Split data temporally
train_df = labeled_data[labeled_data['event_local_time'] <= split_time[0]].copy()
val_df = labeled_data[
    (labeled_data['event_local_time'] > split_time[0]) & 
    (labeled_data['event_local_time'] <= split_time[1])
].copy()
test_df = labeled_data[labeled_data['event_local_time'] > split_time[1]].copy()

print(f"✅ Dataset splitting completed")
print(f"   - Train set: {len(train_df):,} samples ({train_df['event_local_time'].min()} to {train_df['event_local_time'].max()})")
print(f"   - Validation set: {len(val_df):,} samples ({val_df['event_local_time'].min()} to {val_df['event_local_time'].max()})")
print(f"   - Test set: {len(test_df):,} samples ({test_df['event_local_time'].min()} to {test_df['event_local_time'].max()})")

# Check label distribution in each split
for split_name, split_data in [('Train', train_df), ('Validation', val_df), ('Test', test_df)]:
    if len(split_data) > 0:
        label_counts = split_data['label'].value_counts()
        pos_rate = label_counts.get(1, 0) / len(split_data)
        print(f"   - {split_name} positive rate: {pos_rate:.4f}")


### Data Standardization


In [ ]:
print("📊 Standardizing features...")

from sklearn.preprocessing import StandardScaler

# Features to standardize (exclude cyclical and derived features)
features_to_standardize = [
    col for col in extended_feature_cols 
    if not col.endswith('_missing') and col not in ['hour_sin', 'hour_cos', 'month_sin', 'month_cos']
]

scalers = {}

for device in train_df['device_name'].unique():
    # Fit scaler on training data
    device_train = train_df[train_df['device_name'] == device][features_to_standardize]
    scaler = StandardScaler()
    scaler.fit(device_train)
    scalers[device] = scaler
    
    # Apply to train, validation, and test sets
    for df_name, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
        device_mask = df['device_name'] == device
        if device_mask.any():
            df.loc[device_mask, features_to_standardize] = scaler.transform(
                df.loc[device_mask, features_to_standardize]
            )

print(f"✅ Standardization completed for {len(scalers)} devices")
print(f"   - Standardized features: {len(features_to_standardize)}")


## Save Processed Data

Save the processed datasets and configuration for future use.


In [ ]:
print("💾 Saving processed data...")

try:
    # Save processed datasets (same as original data_pipeline.ipynb)
    train_df.to_csv('../dataset/train_data.csv', index=False)
    val_df.to_csv('../dataset/val_data.csv', index=False)
    test_df.to_csv('../dataset/test_data.csv', index=False)
    
    # Save configuration
    dataset_parameters = {
        "feature_cols": extended_feature_cols,
        "window_size": 30,
        "stride": 1,
        "pre_days": pre_days,
        "exclude_periods": [[str(p[0]), str(p[1])] for p in exclude_periods],
        "split_time": [str(split_time[0]), str(split_time[1])],
        "standardized_features": features_to_standardize
    }
    
    with open("../config/dataset_parameters.json", "w") as f:
        json.dump(dataset_parameters, f, indent=4)
    
    print(f"✅ Data saved successfully!")
    print(f"   - Train data: ../dataset/train_data.csv ({len(train_df):,} rows)")
    print(f"   - Validation data: ../dataset/val_data.csv ({len(val_df):,} rows)")
    print(f"   - Test data: ../dataset/test_data.csv ({len(test_df):,} rows)")
    print(f"   - Configuration: ../config/dataset_parameters.json")
    print(f"   - Total features: {len(extended_feature_cols)}")
    
except Exception as e:
    print(f"❌ Error saving data: {e}")


## Summary

This notebook demonstrates the complete data preprocessing pipeline using the refactored package:

1. **Data Loading**: Using modular data loading functions
2. **Data Visualization**: Interactive plots and failure timelines
3. **Data Preprocessing**: 
   - Anomaly detection and removal
   - Missing value imputation
   - Data labeling for pre-failure periods
   - Feature engineering (cyclical encoding, voltage features, temperature delta)
4. **Dataset Splitting**: Temporal train/validation/test split
5. **Data Standardization**: Device-wise feature standardization
6. **Data Saving**: CSV files and configuration for future use

The refactored package provides:
- **Better Organization**: Clear separation of concerns across modules
- **Type Safety**: Comprehensive type hints throughout
- **Error Handling**: Robust validation and informative error messages
- **Documentation**: Extensive docstrings and examples
- **Modularity**: Easy to test, maintain, and extend

The processed data is now ready for model training in separate notebooks.
